
# Assignment 3 — NLP with Transformers (Project Gutenberg)

**Student:** Naimur Rahman Jayed · **Deep Learning Minor · Inholland**

---

## How I organised this notebook

I wrote this notebook as my **main submission**. Each assignment step contains:

1. **Markdown** — what I did, how the code works, and why I made each choice  
2. **Code** — reproducible Python cells  
3. **Visualisations** — plots I use to explain the data and model behaviour  

All figures are also saved under `outputs/figures/` so my teacher can review them without re-running every cell.

| Step | Topic |
|:----:|-------|
| 1 | Data preparation & exploratory plots |
| 2 | Conv1D + BiLSTM + evaluation charts |
| 3 | DistilBERT fine-tuning |
| 4 | Category-conditioned text generation |
| 5 | Final model comparison dashboard |



In [ ]:
# Setup
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display, Image, Markdown

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid", palette="muted")
print("Ready — project root:", ROOT)



---

# Step 1 — Data Preparation

In this step I prepared the Project Gutenberg catalog for deep learning.

## My workflow

1. Loaded **77,070** rows from `pg_catalog.csv`
2. Combined **Title**, **Authors**, and **Subjects** into one input string
3. Parsed **multi-label** bookshelf tags (one book → multiple categories)
4. Selected the **12 most frequent** categories for training on my laptop
5. Cleaned text (lowercase, remove URLs/punctuation)

## What the visualisations below show

| Plot | Purpose |
|------|---------|
| Missing values | I check data quality before modelling |
| Language distribution | I justify filtering to English (`en`) texts |
| Labels per book | I show how multi-label the dataset is |
| Co-occurrence heatmap | I see which categories appear together |
| Class & length histograms | I choose `max_len=96` for padding |
| Word frequencies | I spot dominant tokens in metadata |

**References**  
[1] Inholland, *Minor Deep Learning — Assignment 3*, 2026.  
[2] V. Sanh et al., "DistilBERT," *arXiv:1910.01108*, 2019. https://arxiv.org/abs/1910.01108



In [ ]:
from src.data_loading import load_raw_catalog
from src.multilabel_data import prepare_multilabel_dataset
from src.preprocessing import preprocess_dataframe
from src.visualization import (
    plot_missing_values,
    plot_language_distribution,
    plot_labels_per_book,
    plot_multilabel_cooccurrence,
    plot_class_distribution,
    plot_text_length_distribution,
    plot_word_frequency,
)

raw_df = load_raw_catalog()
print("Shape:", raw_df.shape)
display(raw_df.head())


In [ ]:
# Visualisation 1 — missing values in raw catalog
plot_missing_values(raw_df)


In [ ]:
# Visualisation 2 — languages in full catalog
plot_language_distribution(raw_df)


In [ ]:
# Prepare my working multi-label subset
df, class_names = prepare_multilabel_dataset(raw_df)
df = preprocess_dataframe(df, text_col="text")
df["num_labels"] = df["label_list"].apply(len)
df["category"] = df["label_list"].str[0]
print("Working set:", df.shape, "| categories:", len(class_names))
display(df[["Title", "label_list", "num_labels", "text_clean"]].head(8))


In [ ]:
# Visualisation 3 — multi-label intensity
plot_labels_per_book(df)


In [ ]:
# Visualisation 4 — which categories co-occur?
plot_multilabel_cooccurrence(df["label_list"].tolist(), class_names)


In [ ]:
# Visualisation 5–7 — class balance, text length, vocabulary
plot_class_distribution(df)
plot_text_length_distribution(df, col="text")
tokens = []
for t in df["text_clean"].head(4000):
    tokens.extend(t.split())
plot_word_frequency(tokens, top_n=20)



### Figure interpretation (Step 1)

After running the cells above, I expect to see:

- **Skewed category counts** — *Novels* and *Biographies* dominate; I stratify splits later to handle this.
- **Short texts** — most metadata is under 200 characters, so CNN/LSTM contexts stay small.
- **Co-occurrence blocks** — related shelves (e.g. history + essays) light up together; this explains why multi-label metrics matter.




---

# Step 2 — Conv1D & BiLSTM Multi-label Classification

The assignment requires **two** Keras models. I implemented both with **sigmoid outputs** and `binary_crossentropy` because each book can have **multiple** bookshelf tags.

| Model | Layers I used |
|-------|----------------|
| **Conv1D** | Embedding → Conv1D(128) → GlobalMaxPooling → BatchNorm → Dense → sigmoid |
| **BiLSTM** | Embedding → Bidirectional LSTM(64) → BatchNorm → Dense → sigmoid |

## Metrics

- **F1 micro** — overall label prediction quality  
- **F1 macro** — average per category (fair to rare tags)  
- **Hamming loss** — share of wrong label decisions  
- **AUC** — ranking quality during training  

## Visualisations in this step

1. **Training curves** (loss, AUC, accuracy) — I check overfitting  
2. **Metrics bar chart** — I compare Conv1D vs LSTM side by side  
3. **Per-label F1** — I see which categories are hardest  
4. **Sample prediction cards** — I inspect real mistakes  

**Reference**  
[3] F. Chollet, *Deep Learning with Python*, Manning, 2021.



In [ ]:
from src.multilabel_train import prepare_multilabel_tensors, train_conv1d, train_lstm
from src.multilabel_eval import predict_multilabel
from src.visualization import (
    plot_training_history_multilabel,
    plot_multilabel_metrics_bar,
    plot_per_label_f1,
    plot_sample_predictions,
)

splits, vectorizer, meta = prepare_multilabel_tensors(df)
X_train, X_val, X_test, y_train, y_val, y_test = splits
texts_test = meta["texts_test"]
print("Test samples:", len(X_test))


In [ ]:
# Train Conv1D — I watch validation AUC in the next plot
conv_model, conv_hist, conv_res = train_conv1d(splits, meta, epochs=4)
plot_training_history_multilabel(conv_hist, "conv1d")


In [ ]:
# Train BiLSTM
lstm_model, lstm_hist, lstm_res = train_lstm(splits, meta, epochs=4)
plot_training_history_multilabel(lstm_hist, "lstm")


In [ ]:
# Visualisation — compare test metrics
plot_multilabel_metrics_bar([conv_res, lstm_res])

conv_pred, conv_prob = predict_multilabel(conv_model, X_test)
lstm_pred, lstm_prob = predict_multilabel(lstm_model, X_test)
plot_per_label_f1(y_test, lstm_pred, class_names, name="lstm_per_label_f1.png")
plot_sample_predictions(texts_test, y_test, lstm_prob, class_names, n=4)


In [ ]:
# Table for my report
pd.DataFrame([
    {"Model": "Conv1D", **conv_res["test_metrics"], "sec": conv_res["train_time_sec"]},
    {"Model": "BiLSTM", **lstm_res["test_metrics"], "sec": lstm_res["train_time_sec"]},
])



---

# Step 3 — DistilBERT (Pretrained Transformer)

I fine-tuned **DistilBERT** with PyTorch and Hugging Face `transformers` [2]. I used a **2,000-sample subset** and **1 epoch** on my Mac because full-catalog BERT training is slow without a large GPU.

## Why DistilBERT?

It keeps BERT’s attention mechanism but has fewer layers — a good trade-off for a student laptop.

## What I plot

- Bar chart including BERT in the **model comparison**
- I print JSON metrics for transparency

If I had more time, I would run **3–5 epochs** on the full multi-label set; I expect F1 to rise.



In [ ]:
# Optional: skip if offline — needs HuggingFace download (~250 MB)
RUN_BERT = True  # set False to load saved metrics only

if RUN_BERT:
    from src.bert_classifier import train_bert
    bert_model, bert_tok, bert_res = train_bert()
else:
    bert_res = json.loads((ROOT / "outputs/metrics/bert_multilabel.json").read_text())

display(pd.Series(bert_res["test_metrics"]))


In [ ]:
# Add BERT to comparison chart
all_results = [conv_res, lstm_res, bert_res]
plot_multilabel_metrics_bar(all_results)



---

# Step 4 — Text Generation by Category

I trained a small **character-level LSTM** on metadata from books tagged *Category: History - American* (you can change the category in code).

The assignment judges:

- syntactic plausibility ✓  
- style/category match (checked with my classifier) ✓  
- **not** factual correctness  

The figure below shows my seed text and generated continuation.



In [ ]:
from src.text_generation import train_and_generate
from src.visualization import plot_generated_text_card

gen_model, gen_result = train_and_generate(category="Category: History - American")
plot_generated_text_card(
    gen_result["seed"],
    gen_result["generated_text"],
    gen_result["prompt_category"],
)
print(gen_result["generated_text"])



---

# Final Comparison & Conclusions

## Summary table

I load all saved metrics and plot an **F1 comparison chart**. This is the figure I would discuss in a presentation.

## My conclusions

1. **BiLSTM** was my best classical model on multi-label metadata.  
2. **Conv1D** was faster but slightly weaker — still useful as a baseline.  
3. **DistilBERT** needs more training time before it beats LSTM on my hardware.  
4. **Text generation** works as a demo; a full Transformer decoder would be the next step [4].

## Ethics

Gutenberg metadata reflects historical bias. I would not deploy this without human review.

**Reference**  
[4] A. Vaswani et al., "Attention is all you need," *NeurIPS*, 2017. https://arxiv.org/abs/1706.03762



In [ ]:
# Final dashboard — all saved metrics
from src.visualization import plot_multilabel_metrics_bar

metrics_dir = ROOT / "outputs" / "metrics"
loaded = []
for path in sorted(metrics_dir.glob("*multilabel.json")):
    data = json.loads(path.read_text())
    if "test_metrics" in data:
        loaded.append(data)
display(pd.DataFrame([
    {"model": d["model"], **d["test_metrics"], "train_sec": d.get("train_time_sec")}
    for d in loaded
]))
if loaded:
    plot_multilabel_metrics_bar(loaded)

# Show saved figures gallery
fig_dir = ROOT / "outputs" / "figures"
for img in sorted(fig_dir.glob("*.png")):
    display(Markdown(f"**{img.name}**"))
    display(Image(filename=str(img), width=700))
